

# Week 03 — Data Contract: CTR / Engagement Opportunity Scoring
**Lane:** CTR / Engagement Opportunity Scoring
**Dataset:** FlyRank/internship-warehouse (dim_content + fact_content_daily_performance_sample)

This notebook is the data contract for my lane, verified live against the real warehouse release.
Every claim in a markdown cell is checked by the query cell directly below it.
```



In [1]:
!pip install duckdb --upgrade -q
import duckdb

con = duckdb.connect()
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")
print("DuckDB version:", duckdb.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 67.8 MB/s eta 0:00:00
DuckDB version: 1.5.4


The next cell reads the token from Colab's secret store at runtime only — it never gets written
into the `.ipynb` file, so committing this notebook afterward is safe.

In [2]:
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

con.sql(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE huggingface,
    TOKEN '{hf_token}'
);
""")

con.sql("""
CREATE OR REPLACE VIEW dim_content AS
SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet');
""")
con.sql("""
CREATE OR REPLACE VIEW fact_daily AS
SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet');
""")
con.sql("""
CREATE OR REPLACE VIEW dim_clients AS
SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet');
""")

print("fact_daily columns:")
con.sql("DESCRIBE fact_daily;").show()
print("dim_content columns:")
con.sql("DESCRIBE dim_content;").show()

fact_daily columns:
┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │

## 1. Unit of analysis + time window

**Unit of analysis:** the native grain of `fact_content_daily_performance` is one row per
`content_hash_id` × `client_hash_id` × `report_date`. For the CTR opportunity score, I aggregate
this up to one row per `(client_hash_id, content_hash_id)`, summed/averaged across the sample
window, then join `dim_content` for content metadata.

**Time window:** the file is documented as "the latest full month" — verified below rather than assumed.

**Claims to verify:** (a) the daily grain has no duplicate keys, (b) the date range is one contiguous month.

In [3]:
# (a) grain check — should return 0 rows
con.sql("""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
FROM fact_daily
GROUP BY 1, 2, 3
HAVING COUNT(*) > 1
""").show()

# (b) date window
con.sql("""
SELECT MIN(report_date) AS min_date,
       MAX(report_date) AS max_date,
       COUNT(DISTINCT report_date) AS n_distinct_days
FROM fact_daily
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬─────────────────────────┬──────────────────────────┬───────┐
│ report_date │     client_hash_id      │     content_hash_id      │   n   │
│    date     │         varchar         │         varchar          │ int64 │
├─────────────┼─────────────────────────┼──────────────────────────┼───────┤
│ 2026-06-13  │ client_e00b29e582949543 │ content_e3491394a9f3e2b3 │     2 │
│ 2026-06-13  │ client_e00b29e582949543 │ content_83b20e2cd4d5437f │     2 │
│ 2026-06-13  │ client_e00b29e582949543 │ content_04307a138472fc03 │     2 │
│ 2026-06-13  │ client_e00b29e582949543 │ content_638ac5ad8023d177 │     2 │
│ 2026-06-13  │ client_e00b29e582949543 │ content_ff881e76e85001f4 │     2 │
│ 2026-06-13  │ client_810019792c9b8efc │ content_097eada60a3a3f2b │     2 │
│ 2026-06-13  │ client_810019792c9b8efc │ content_fa60e9dea96f683a │     2 │
│ 2026-06-13  │ client_810019792c9b8efc │ content_dacee0e0bac2dd31 │     2 │
│ 2026-06-13  │ client_810019792c9b8efc │ content_034d4ed04b3ddd55 │     2 │

## 2. Fields: feature / label / context / excluded

| Bucket | Fields | Why |
|---|---|---|
| **Feature** | `impressions`, `clicks`, `avg_position`, `sessions`, `engagement_rate`, `scroll_rate` (from `fact_daily`, aggregated); `content_type`, `main_intent`, `word_count` (from `dim_content`) | Observed signals + derived measurements, available before any decision point |
| **Label / proxy** | Derived **CTR gap** = observed CTR − expected CTR for that row's position tier (expected = median CTR within the tier, computed from this same data) | This lane ranks opportunity rather than predicting a ground-truth outcome |
| **Context-only** | `client_hash_id`, `content_hash_id`, `report_date`, `dim_clients.gsc_data_start`, `dim_clients.ga4_data_start` | Join keys and coverage flags — never modeling inputs |
| **Excluded** | `health_score`, `priority_score`, `action_type`, refresh flags; any raw URL/query/title/client-name field | Product decisions and raw-origin fields — not shipped here, would leak the answer if they were |

**Claim to verify:** the excluded/product fields genuinely aren't present in this release.

In [4]:
excluded_candidates = ['health_score', 'priority_score', 'action_type', 'refresh_tier',
                        'is_quick_win', 'needs_ctr_fix', 'url', 'raw_url', 'query',
                        'client_name', 'title']

fact_cols = [r[0] for r in con.sql("DESCRIBE fact_daily;").fetchall()]
dim_cols  = [r[0] for r in con.sql("DESCRIBE dim_content;").fetchall()]
all_cols = set(fact_cols) | set(dim_cols)

present = [c for c in excluded_candidates if c in all_cols]
print("Excluded/product fields actually present (should be empty list):", present)

Excluded/product fields actually present (should be empty list): []


## 3. Verification: counts and missing values

**Claim:** row/entity counts are sane, and missing-value rates are directionally consistent with
the lane guide's documented density figures (GSC impressions >> GA4 sessions >> scroll events,
across the full 78.8M-row history — I expect the same *proportions* here, not the same counts,
since this is a single month).

In [6]:
con.sql("""
SELECT
    COUNT(*)                                   AS total_rows,
    COUNT(DISTINCT content_hash_id)            AS distinct_content,
    COUNT(DISTINCT client_hash_id)             AS distinct_clients,
    ROUND(100.0 * COUNT(*) FILTER (WHERE gsc_impressions IS NOT NULL AND gsc_impressions > 0) / COUNT(*), 2) AS pct_with_impressions,
    ROUND(100.0 * COUNT(*) FILTER (WHERE gsc_clicks IS NOT NULL AND gsc_clicks > 0) / COUNT(*), 2)            AS pct_with_clicks,
    ROUND(100.0 * COUNT(*) FILTER (WHERE ga4_sessions IS NOT NULL AND ga4_sessions > 0) / COUNT(*), 2)        AS pct_with_sessions,
    ROUND(100.0 * COUNT(*) FILTER (WHERE scroll_events IS NOT NULL) / COUNT(*), 2)                       AS pct_with_scroll,
    ROUND(100.0 * COUNT(*) FILTER (WHERE gsc_avg_position IS NULL) / COUNT(*), 2)                          AS pct_missing_position
FROM fact_daily
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬──────────────────┬──────────────────┬──────────────────────┬─────────────────┬───────────────────┬─────────────────┬──────────────────────┐
│ total_rows │ distinct_content │ distinct_clients │ pct_with_impressions │ pct_with_clicks │ pct_with_sessions │ pct_with_scroll │ pct_missing_position │
│   int64    │      int64       │      int64       │        double        │     double      │      double       │     double      │        double        │
├────────────┼──────────────────┼──────────────────┼──────────────────────┼─────────────────┼───────────────────┼─────────────────┼──────────────────────┤
│   11694072 │           409205 │               65 │                33.17 │            3.83 │              5.38 │            79.5 │                66.83 │
└────────────┴──────────────────┴──────────────────┴──────────────────────┴─────────────────┴───────────────────┴─────────────────┴──────────────────────┘



**Claim:** every `content_hash_id` in the daily fact table joins cleanly to `dim_content` — no orphaned rows.

In [7]:
con.sql("""
SELECT COUNT(*) AS orphaned_rows
FROM fact_daily f
LEFT JOIN dim_content d ON f.content_hash_id = d.content_hash_id
WHERE d.content_hash_id IS NULL
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────┐
│ orphaned_rows │
│     int64     │
├───────────────┤
│             0 │
└───────────────┘



### Building the lane's analysis table

Aggregating `fact_daily` to one row per `(client_hash_id, content_hash_id)`, then joining content
metadata. This is the table the CTR opportunity score is actually computed on.

In [9]:
con.sql("""
CREATE OR REPLACE VIEW content_ctr_base AS
SELECT
    f.client_hash_id,
    f.content_hash_id,
    SUM(f.gsc_impressions)     AS impressions_sum,
    SUM(f.gsc_clicks)          AS clicks_sum,
    AVG(f.gsc_avg_position)    AS avg_position,
    SUM(f.ga4_sessions)        AS sessions_sum,
    -- AVG(f.engagement_rate) AS avg_engagement_rate, -- Column 'engagement_rate' not found in fact_daily
    d.content_type,
    d.main_intent,
    d.word_count
FROM fact_daily f
JOIN dim_content d ON f.content_hash_id = d.content_hash_id
GROUP BY 1, 2, d.content_type, d.main_intent, d.word_count
""")

# grain check on the new table — should be 0 duplicates
con.sql("""
SELECT client_hash_id, content_hash_id, COUNT(*) AS n
FROM content_ctr_base
GROUP BY 1, 2
HAVING COUNT(*) > 1
""").show()

con.sql("SELECT COUNT(*) AS total_content_client_rows FROM content_ctr_base").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┬─────────────────┬───────┐
│ client_hash_id │ content_hash_id │   n   │
│    varchar     │     varchar     │ int64 │
└────────────────┴─────────────────┴───────┘
                   0 rows                 



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────────────────┐
│ total_content_client_rows │
│           int64           │
├───────────────────────────┤
│                    409205 │
└───────────────────────────┘



### Deriving the CTR-gap proxy

Position tiers bucketed as `1-3 / 4-10 / 11-20 / 20+`. Expected CTR per tier = median observed CTR
within that tier, from this same window. Gap = observed CTR − expected CTR for its tier; negative
gaps are the opportunity candidates.

**Claim:** enough rows clear a minimum-volume floor (`impressions_sum >= 100`) that the gap isn't
dominated by noise.

In [10]:
con.sql("""
SELECT
    COUNT(*) FILTER (WHERE impressions_sum >= 100) AS rows_impr_ge_100,
    COUNT(*) FILTER (WHERE impressions_sum >= 500) AS rows_impr_ge_500,
    COUNT(*)                                        AS total_rows
FROM content_ctr_base
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────┬──────────────────┬────────────┐
│ rows_impr_ge_100 │ rows_impr_ge_500 │ total_rows │
│      int64       │      int64       │   int64    │
├──────────────────┼──────────────────┼────────────┤
│           101917 │            52766 │     409205 │
└──────────────────┴──────────────────┴────────────┘



In [11]:
con.sql("""
CREATE OR REPLACE VIEW content_ctr_scored AS
WITH tiered AS (
    SELECT *,
        clicks_sum::DOUBLE / NULLIF(impressions_sum, 0) AS ctr,
        CASE
            WHEN avg_position <= 3  THEN '1-3'
            WHEN avg_position <= 10 THEN '4-10'
            WHEN avg_position <= 20 THEN '11-20'
            ELSE '20+'
        END AS position_tier
    FROM content_ctr_base
    WHERE impressions_sum >= 100
),
expected AS (
    SELECT position_tier, MEDIAN(ctr) AS expected_ctr
    FROM tiered
    GROUP BY position_tier
)
SELECT t.*, e.expected_ctr, (t.ctr - e.expected_ctr) AS ctr_gap
FROM tiered t
JOIN expected e USING (position_tier)
""")

con.sql("""
SELECT position_tier, expected_ctr, COUNT(*) AS n
FROM content_ctr_scored
GROUP BY 1, 2
ORDER BY 1
""").show()

# top opportunity candidates — biggest negative gap
con.sql("""
SELECT client_hash_id, content_hash_id, impressions_sum, ctr, position_tier, expected_ctr, ctr_gap
FROM content_ctr_scored
ORDER BY ctr_gap ASC
LIMIT 10
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────┬───────────────────────┬───────┐
│ position_tier │     expected_ctr      │   n   │
│    varchar    │        double         │ int64 │
├───────────────┼───────────────────────┼───────┤
│ 1-3           │  0.003778337531486146 │  1887 │
│ 11-20         │  0.003115264797507788 │ 26525 │
│ 20+           │                   0.0 │ 31684 │
│ 4-10          │ 0.0037656903765690376 │ 41821 │
└───────────────┴───────────────────────┴───────┘



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────────────┬──────────────────────────┬─────────────────┬────────┬───────────────┬──────────────────────┬───────────────────────┐
│     client_hash_id      │     content_hash_id      │ impressions_sum │  ctr   │ position_tier │     expected_ctr     │        ctr_gap        │
│         varchar         │         varchar          │     int128      │ double │    varchar    │        double        │        double         │
├─────────────────────────┼──────────────────────────┼─────────────────┼────────┼───────────────┼──────────────────────┼───────────────────────┤
│ client_e00b29e582949543 │ content_f7600a29c4e02c5f │            2048 │    0.0 │ 1-3           │ 0.003778337531486146 │ -0.003778337531486146 │
│ client_e00b29e582949543 │ content_21b836ee3e588934 │             225 │    0.0 │ 1-3           │ 0.003778337531486146 │ -0.003778337531486146 │
│ client_e00b29e582949543 │ content_42763b2fdef41d9f │             126 │    0.0 │ 1-3           │ 0.003778337531486146 │ -0.003778